In [2]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 29.3 MB/s eta 0:00:00


In [4]:
import os, json, time
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata, drive

login(token=userdata.get('HF_LLM'))
drive.mount('/content/drive')

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_NAME = "llama_3.1_8b"
DRIVE_PATH = '/content/drive/MyDrive/derivation_boundaries'
RESULTS_PATH = f"{DRIVE_PATH}/airline_results_{MODEL_NAME}.json"
LOGIT_LENS_PATH = f"{DRIVE_PATH}/logit_lens_{MODEL_NAME}.json"

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Results file exists:", os.path.exists(RESULTS_PATH))

Mounted at /content/drive
CUDA: True
GPU: Tesla T4
Results file exists: False


In [5]:
from google.colab import files
uploaded = files.upload()

Saving airline_results_llama_3.1_8b.json to airline_results_llama_3.1_8b.json


In [6]:
import json

RESULTS_PATH = '/content/airline_results_llama_3.1_8b.json'

with open(RESULTS_PATH) as f:
    results = json.load(f)

print(f"Loaded {len(results)} problems")
print(f"Keys in first result: {list(results[0].keys())}")

Loaded 60 problems
Keys in first result: ['problem_idx', 'model', 'domain', 'prompt', 'ground_truth', 'gt_info', 'condition_a_response', 'condition_a_predicted', 'condition_a_correct', 'condition_b_response', 'condition_b_predicted', 'condition_b_correct']


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto",
)
model.eval()
print("Model loaded.")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded.


In [8]:
def get_logit_lens(prompt_text, model, tokenizer):
    """
    For a given prompt, captures hidden states at every layer
    and projects them to vocabulary space to get per-layer predictions.

    Returns:
        tokens: list of token strings in the response
        layer_predictions: list of lists — layer_predictions[layer][token_pos] = predicted token string
        layer_probs: list of lists — layer_probs[layer][token_pos] = probability of top predicted token
    """
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True,  # this is the key line
        )

    # outputs.hidden_states is a tuple of 33 tensors
    # index 0 = embedding layer, index 1-32 = transformer layers
    hidden_states = outputs.hidden_states  # tuple of (1, seq_len, hidden_dim)

    # Get the final layer norm from the model
    final_layer_norm = model.model.norm
    lm_head = model.lm_head

    layer_predictions = []
    layer_probs = []

    for layer_idx in range(1, 33):  # layers 1-32
        hs = hidden_states[layer_idx]  # (1, seq_len, hidden_dim)

        # Apply final layer norm and project to vocabulary
        with torch.no_grad():
            normed = final_layer_norm(hs)
            logits = lm_head(normed)  # (1, seq_len, vocab_size)

        # Get top prediction at each position
        probs = torch.softmax(logits[0], dim=-1)  # (seq_len, vocab_size)
        top_tokens = torch.argmax(probs, dim=-1)  # (seq_len,)
        top_probs = probs[range(len(top_tokens)), top_tokens]  # (seq_len,)

        # Convert to strings
        pred_strings = [tokenizer.decode([t.item()]) for t in top_tokens]
        prob_values = [p.item() for p in top_probs]

        layer_predictions.append(pred_strings)
        layer_probs.append(prob_values)

    # Get actual tokens in the full sequence
    all_tokens = [tokenizer.decode([t.item()]) for t in inputs["input_ids"][0]]

    return all_tokens, layer_predictions, layer_probs, input_len

In [9]:
# Test logit lens on problem 0 baseline prompt
import os, json, re

if not os.path.exists("RuleArena"):
    os.system("git clone -q https://github.com/SkyRiver-2000/RuleArena.git")

os.chdir("/content/RuleArena/airline")

exec(open("compute_answer.py").read().split("if __name__")[0])
check_base_tables = load_checking_fee()

with open("reference_rules.txt") as f:
    REFERENCE_RULES = f.read()

problems = []
with open("synthesized_problems/comp_0.jsonl") as f:
    for line in f:
        problems.append(json.loads(line))

print(f"Loaded {len(problems)} problems")

Loaded 100 problems


In [10]:
def build_baseline_prompt(problem):
    system = (
        "You are an airline pricing assistant. You will be given the policies of "
        "American Airlines and a passenger's itinerary and baggage details. "
        "Compute the TOTAL COST (flight ticket + all checked bag fees, including "
        "any oversize or overweight charges) according to the policies. "
        "Show your step-by-step reasoning for each bag, then give the final answer "
        "as: FINAL TOTAL: $<amount>"
    )
    user = f"POLICIES:\n{REFERENCE_RULES}\n\nPASSENGER:\n{problems[0]['prompt']}"
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    return prompt_text

prompt_text = build_baseline_prompt(problems[0])
print(f"Prompt length: {len(prompt_text)} characters")
print(f"Prompt tokens: {len(tokenizer(prompt_text)['input_ids'])}")

Prompt length: 20569 characters
Prompt tokens: 3858


In [11]:
# Take saved response from results and build full prompt+response text
problem_0_response_a = results[0]['condition_a_response']

# Full text = prompt + response
full_text = prompt_text + problem_0_response_a

print(f"Full text tokens: {len(tokenizer(full_text)['input_ids'])}")
print(f"Response preview: {problem_0_response_a[:300]}")

Full text tokens: 4232
Response preview: To calculate the total cost, we need to determine the fees for each bag and then add them to the flight ticket cost.

1. The first bag (backpack: 22 x 13 x 6 inches, 10 lbs) is within the size and weight limits for Main Cabin Class. The fee for the first bag is $40.

2. The second bag (luggage box: 


In [12]:
print("Running logit lens on problem 0 baseline response...")
print("This will take ~30-60 seconds...")

all_tokens, layer_predictions, layer_probs, input_len = get_logit_lens(full_text, model, tokenizer)

print(f"Total tokens in sequence: {len(all_tokens)}")
print(f"Input length: {input_len}")
print(f"Response tokens: {len(all_tokens) - input_len}")
print(f"Number of layers captured: {len(layer_predictions)}")

Running logit lens on problem 0 baseline response...
This will take ~30-60 seconds...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Total tokens in sequence: 4232
Input length: 4232
Response tokens: 0
Number of layers captured: 32


In [13]:
# Tokenize prompt alone to get input length
prompt_tokens = tokenizer(prompt_text, return_tensors="pt")
input_len = prompt_tokens["input_ids"].shape[-1]

# Tokenize full text (prompt + response)
full_tokens = tokenizer(full_text, return_tensors="pt")
total_len = full_tokens["input_ids"].shape[-1]

print(f"Prompt tokens: {input_len}")
print(f"Total tokens: {total_len}")
print(f"Response tokens: {total_len - input_len}")

Prompt tokens: 3858
Total tokens: 4232
Response tokens: 374


In [14]:
def get_logit_lens(full_text, model, tokenizer):
    """
    Runs a single forward pass with output_hidden_states=True.
    Projects each layer's hidden state to vocabulary space.
    Returns per-layer predictions for every token position.
    """
    inputs = tokenizer(full_text, return_tensors="pt").to(model.device)
    total_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True,
        )

    hidden_states = outputs.hidden_states  # 33 tensors: embedding + 32 layers
    final_layer_norm = model.model.norm
    lm_head = model.lm_head

    layer_predictions = []
    layer_probs = []

    for layer_idx in range(1, 33):
        hs = hidden_states[layer_idx]  # (1, seq_len, hidden_dim)

        with torch.no_grad():
            normed = final_layer_norm(hs)
            logits = lm_head(normed)  # (1, seq_len, vocab_size)

        probs = torch.softmax(logits[0], dim=-1)
        top_tokens = torch.argmax(probs, dim=-1)
        top_probs = probs[range(len(top_tokens)), top_tokens]

        pred_strings = [tokenizer.decode([t.item()]) for t in top_tokens]
        prob_values = [p.item() for p in top_probs]

        layer_predictions.append(pred_strings)
        layer_probs.append(prob_values)

    # Get actual tokens
    all_tokens = [tokenizer.decode([t.item()]) for t in inputs["input_ids"][0]]

    return all_tokens, layer_predictions, layer_probs, total_len

# Now run it with correct input_len tracking
prompt_tokens = tokenizer(prompt_text, return_tensors="pt")
input_len = prompt_tokens["input_ids"].shape[-1]

print("Running logit lens...")
all_tokens, layer_predictions, layer_probs, total_len = get_logit_lens(full_text, model, tokenizer)

response_tokens = all_tokens[input_len:]
print(f"Prompt tokens: {input_len}")
print(f"Response tokens: {len(response_tokens)}")
print(f"First 20 response tokens: {response_tokens[:20]}")

Running logit lens...
Prompt tokens: 3858
Response tokens: 374
First 20 response tokens: ['To', ' calculate', ' the', ' total', ' cost', ',', ' we', ' need', ' to', ' determine', ' the', ' fees', ' for', ' each', ' bag', ' and', ' then', ' add', ' them', ' to']


In [15]:
# Find token positions where the model generates dollar amounts
for i, token in enumerate(response_tokens):
    if '$' in token or any(c.isdigit() for c in token):
        print(f"Position {input_len + i}: '{token}'")

Position 3883: '1'
Position 3893: '22'
Position 3896: '13'
Position 3899: '6'
Position 3903: '10'
Position 3925: ' $'
Position 3926: '40'
Position 3928: '2'
Position 3939: '44'
Position 3942: '22'
Position 3945: '20'
Position 3949: '69'
Position 3977: ' $'
Position 3978: '45'
Position 3980: '3'
Position 3991: '34'
Position 3994: '18'
Position 3997: '12'
Position 4001: '51'
Position 4028: ' $'
Position 4029: '150'
Position 4031: '4'
Position 4041: '38'
Position 4044: '22'
Position 4047: '16'
Position 4051: '84'
Position 4064: ' $'
Position 4065: '100'
Position 4069: '53'
Position 4073: '24'
Position 4078: '70'
Position 4082: '32'
Position 4086: '5'
Position 4096: '38'
Position 4099: '14'
Position 4102: '11'
Position 4106: '90'
Position 4120: ' $'
Position 4121: '100'
Position 4125: '53'
Position 4129: '24'
Position 4134: '70'
Position 4138: '32'
Position 4154: ' $'
Position 4155: '180'
Position 4161: ' $'
Position 4162: '40'
Position 4168: ' $'
Position 4169: '45'
Position 4175: ' $'
Po

In [16]:
# Look at position 3978 — where model generates $45 for bag 2
# But correct answer should involve oversize/overweight fees
pos = 3978  # the "45" token

print(f"Actual token at position {pos}: '{all_tokens[pos]}'")
print(f"\nPer-layer predictions at this position:")
print(f"{'Layer':>6} | {'Predicted':>15} | {'Probability':>12}")
print("-" * 40)
for layer_idx in range(32):
    pred = layer_predictions[layer_idx][pos]
    prob = layer_probs[layer_idx][pos]
    print(f"{layer_idx+1:>6} | {repr(pred):>15} | {prob:>12.4f}")

Actual token at position 3978: '45'

Per-layer predictions at this position:
 Layer |       Predicted |  Probability
----------------------------------------
     1 |           'kyt' |       0.0190
     2 |        'edList' |       0.0199
     3 |           'kyt' |       0.0330
     4 |         'alnız' |       0.0864
     5 |         'alnız' |       0.0297
     6 |            'ęk' |       0.0142
     7 |            'ęk' |       0.0183
     8 |            'ー�' |       0.0103
     9 |            'ải' |       0.0059
    10 |           'hof' |       0.0491
    11 | 'ComputedStyle' |       0.0068
    12 |        'elters' |       0.0124
    13 |        'elters' |       0.0251
    14 |          'utex' |       0.0265
    15 |        'sonian' |       0.0325
    16 |         'uliar' |       0.0172
    17 |    ' lesbische' |       0.0086
    18 |          'imir' |       0.0082
    19 |     'omination' |       0.0165
    20 |      '-toggler' |       0.0142
    21 |        'iggers' |       0.0201
  

In [17]:
# Look at position 3977 — the '$' token before '45'
pos = 3977

print(f"Actual token at position {pos}: '{all_tokens[pos]}'")
print(f"\nPer-layer predictions at this position:")
print(f"{'Layer':>6} | {'Predicted':>15} | {'Probability':>12}")
print("-" * 40)
for layer_idx in range(32):
    pred = layer_predictions[layer_idx][pos]
    prob = layer_probs[layer_idx][pos]
    print(f"{layer_idx+1:>6} | {repr(pred):>15} | {prob:>12.4f}")

Actual token at position 3977: ' $'

Per-layer predictions at this position:
 Layer |       Predicted |  Probability
----------------------------------------
     1 |          'erli' |       0.0059
     2 |         'aData' |       0.0120
     3 |          'erli' |       0.0131
     4 |          ')did' |       0.0217
     5 |         '/***/' |       0.0334
     6 |           'atk' |       0.0123
     7 |   '.Accessible' |       0.0096
     8 |           'PTH' |       0.0039
     9 |           'ايل' |       0.0057
    10 |            'xD' |       0.0122
    11 |         'immel' |       0.0287
    12 |            '�回' |       0.0079
    13 |          'utex' |       0.0060
    14 |          'utex' |       0.0280
    15 |          'ương' |       0.0104
    16 |         '=<?=$' |       0.0160
    17 |         'undry' |       0.0172
    18 |           '#af' |       0.0247
    19 |           '#af' |       0.0206
    20 |           '#af' |       0.1064
    21 |            '45' |       0.0129
  

In [18]:
pos = 3925  # the '$' before '40' for bag 1

print(f"Actual token at position {pos}: '{all_tokens[pos]}'")
print(f"\nPer-layer predictions at this position:")
print(f"{'Layer':>6} | {'Predicted':>15} | {'Probability':>12}")
print("-" * 40)
for layer_idx in range(32):
    pred = layer_predictions[layer_idx][pos]
    prob = layer_probs[layer_idx][pos]
    print(f"{layer_idx+1:>6} | {repr(pred):>15} | {prob:>12.4f}")

Actual token at position 3925: ' $'

Per-layer predictions at this position:
 Layer |       Predicted |  Probability
----------------------------------------
     1 |          'erli' |       0.0045
     2 |         'aData' |       0.0183
     3 |           '#ad' |       0.0118
     4 |          ')did' |       0.0161
     5 |         'undry' |       0.0156
     6 |         'undry' |       0.0074
     7 |         'udent' |       0.0112
     8 |         'udent' |       0.0099
     9 |   '_trampoline' |       0.0048
    10 |             '胎' |       0.0063
    11 |         'immel' |       0.0181
    12 |            'uz' |       0.0214
    13 |          'iyan' |       0.0183
    14 |         '/***/' |       0.0388
    15 |       'ardware' |       0.0157
    16 |          'iani' |       0.0084
    17 |          'iani' |       0.0461
    18 |          'iani' |       0.0131
    19 |           '#af' |       0.0344
    20 |           '#af' |       0.0693
    21 |           '#af' |       0.0107
  

In [19]:
pos = 3924

print(f"Actual token at position {pos}: '{all_tokens[pos]}'")
print(f"\nPer-layer predictions at this position:")
print(f"{'Layer':>6} | {'Predicted':>15} | {'Probability':>12}")
print("-" * 40)
for layer_idx in range(32):
    pred = layer_predictions[layer_idx][pos]
    prob = layer_probs[layer_idx][pos]
    print(f"{layer_idx+1:>6} | {repr(pred):>15} | {prob:>12.4f}")

Actual token at position 3924: ' is'

Per-layer predictions at this position:
 Layer |       Predicted |  Probability
----------------------------------------
     1 |         'otope' |       0.0674
     2 |         'otope' |       0.0576
     3 |         'otope' |       0.0165
     4 |           "'gc" |       0.0219
     5 |         '/***/' |       0.0190
     6 |          'Muon' |       0.0107
     7 |         ' Pact' |       0.0090
     8 |        ' subur' |       0.0094
     9 |        ' subur' |       0.0315
    10 |          'ries' |       0.0097
    11 |          'zell' |       0.0138
    12 |        'herits' |       0.0118
    13 |         'ilers' |       0.0149
    14 |           "'gc" |       0.0242
    15 |         'alian' |       0.0145
    16 |        'herits' |       0.0315
    17 |        'herits' |       0.0134
    18 |        '-alist' |       0.0146
    19 |        '-alist' |       0.0133
    20 |    ' therefore' |       0.0593
    21 |    ' therefore' |       0.0781
 

In [20]:
# Find all ' $' positions in the response
dollar_positions = []
for i in range(input_len, len(all_tokens)):
    if all_tokens[i] == ' $':
        # What does the model predict here (the fee value)
        final_pred = layer_predictions[31][i]  # layer 32 prediction
        dollar_positions.append({
            'position': i,
            'layer32_pred': final_pred,
            'actual_next': all_tokens[i+1] if i+1 < len(all_tokens) else None
        })

print(f"Found {len(dollar_positions)} dollar sign positions in response")
print(f"\n{'Position':>10} | {'Layer32 pred':>15} | {'Actual next':>15}")
print("-" * 50)
for d in dollar_positions:
    print(f"{d['position']:>10} | {repr(d['layer32_pred']):>15} | {repr(d['actual_next']):>15}")

Found 21 dollar sign positions in response

  Position |    Layer32 pred |     Actual next
--------------------------------------------------
      3925 |            '35' |            '40'
      3977 |            '45' |            '45'
      4028 |           '150' |           '150'
      4064 |           '100' |           '100'
      4120 |           '100' |           '100'
      4154 |           '180' |           '180'
      4161 |            '40' |            '40'
      4168 |            '45' |            '45'
      4175 |           '150' |           '150'
      4182 |           '100' |           '100'
      4190 |           '100' |           '100'
      4197 |           '435' |            '40'
      4200 |            '45' |            '45'
      4203 |           '150' |           '150'
      4206 |           '100' |           '100'
      4209 |           '100' |           '100'
      4212 |           '435' |           '435'
      4218 |           '180' |           '180'
      4221 |

In [21]:
def find_commit_layer(position, layer_predictions, layer_probs, threshold=0.5):
    """Find the first layer where prediction exceeds threshold confidence."""
    final_pred = layer_predictions[31][position]  # what layer 32 predicts

    for layer_idx in range(32):
        pred = layer_predictions[layer_idx][position]
        prob = layer_probs[layer_idx][position]
        if pred == final_pred and prob >= threshold:
            return layer_idx + 1  # 1-indexed

    return 32  # if never reaches threshold, commit at final layer

print(f"{'Position':>10} | {'Final pred':>12} | {'Actual':>12} | {'Commit layer':>12}")
print("-" * 55)
for d in dollar_positions:
    pos = d['position']
    commit = find_commit_layer(pos, layer_predictions, layer_probs)
    print(f"{pos:>10} | {repr(d['layer32_pred']):>12} | {repr(d['actual_next']):>12} | {commit:>12}")

  Position |   Final pred |       Actual | Commit layer
-------------------------------------------------------
      3925 |         '35' |         '40' |           24
      3977 |         '45' |         '45' |           23
      4028 |        '150' |        '150' |           26
      4064 |        '100' |        '100' |           28
      4120 |        '100' |        '100' |           25
      4154 |        '180' |        '180' |           25
      4161 |         '40' |         '40' |           25
      4168 |         '45' |         '45' |           24
      4175 |        '150' |        '150' |           24
      4182 |        '100' |        '100' |           24
      4190 |        '100' |        '100' |           25
      4197 |        '435' |         '40' |           28
      4200 |         '45' |         '45' |           24
      4203 |        '150' |        '150' |           25
      4206 |        '100' |        '100' |           24
      4209 |        '100' |        '100' |      

In [22]:
# Classify positions manually for problem 0
boundary_positions = [4064, 4120, 4182, 4190, 4206, 4209]  # overweight/oversize fee lookups
elementary_positions = [3977, 4028, 4154, 4161, 4168, 4175]  # base fees, ticket price

boundary_commits = [find_commit_layer(p, layer_predictions, layer_probs) for p in boundary_positions]
elementary_commits = [find_commit_layer(p, layer_predictions, layer_probs) for p in elementary_positions]

print(f"Boundary step commit layers: {boundary_commits}")
print(f"Elementary step commit layers: {elementary_commits}")
print(f"Mean boundary commit layer: {np.mean(boundary_commits):.1f}")
print(f"Mean elementary commit layer: {np.mean(elementary_commits):.1f}")

Boundary step commit layers: [28, 25, 24, 25, 24, 25]
Elementary step commit layers: [23, 26, 25, 25, 24, 24]
Mean boundary commit layer: 25.2
Mean elementary commit layer: 24.5


In [23]:
def get_input_numbers(problem):
    """Extract all numbers that appear in the problem input."""
    input_numbers = set()

    # Ticket/base price
    input_numbers.add(str(problem["info"]["base_price"]))

    # Bag dimensions and weights
    for bag in problem["info"]["bag_list"]:
        for dim in bag["size"]:
            input_numbers.add(str(dim))
        input_numbers.add(str(bag["weight"]))
        input_numbers.add(str(bag["id"]))
        # Also add dimension sums - elementary computation
        input_numbers.add(str(sum(bag["size"])))

    return input_numbers

# Test on problem 0
input_nums = get_input_numbers(problems[0])
print(f"Input numbers for problem 0: {sorted(input_nums)}")

Input numbers for problem 0: ['1', '10', '11', '12', '13', '14', '16', '18', '180', '2', '20', '22', '3', '34', '38', '4', '41', '44', '5', '51', '6', '63', '64', '69', '76', '84', '86', '90']


In [24]:
# Classify each dollar position as boundary or elementary
print(f"{'Position':>10} | {'Predicted fee':>15} | {'Type':>12}")
print("-" * 45)
for d in dollar_positions:
    pos = d['position']
    pred = d['layer32_pred']
    # Strip spaces from predicted token
    pred_clean = pred.strip()
    if pred_clean in input_nums:
        classification = 'ELEMENTARY'
    else:
        classification = 'BOUNDARY'
    commit = find_commit_layer(pos, layer_predictions, layer_probs)
    print(f"{pos:>10} | {repr(pred):>15} | {classification:>12} | layer {commit}")

  Position |   Predicted fee |         Type
---------------------------------------------
      3925 |            '35' |     BOUNDARY | layer 24
      3977 |            '45' |     BOUNDARY | layer 23
      4028 |           '150' |     BOUNDARY | layer 26
      4064 |           '100' |     BOUNDARY | layer 28
      4120 |           '100' |     BOUNDARY | layer 25
      4154 |           '180' |   ELEMENTARY | layer 25
      4161 |            '40' |     BOUNDARY | layer 25
      4168 |            '45' |     BOUNDARY | layer 24
      4175 |           '150' |     BOUNDARY | layer 24
      4182 |           '100' |     BOUNDARY | layer 24
      4190 |           '100' |     BOUNDARY | layer 25
      4197 |           '435' |     BOUNDARY | layer 28
      4200 |            '45' |     BOUNDARY | layer 24
      4203 |           '150' |     BOUNDARY | layer 25
      4206 |           '100' |     BOUNDARY | layer 24
      4209 |           '100' |     BOUNDARY | layer 25
      4212 |           '435' |

In [25]:
boundary_commits = []
elementary_commits = []

for d in dollar_positions:
    pos = d['position']
    pred = d['layer32_pred'].strip()
    commit = find_commit_layer(pos, layer_predictions, layer_probs)
    if pred in input_nums:
        elementary_commits.append(commit)
    else:
        boundary_commits.append(commit)

print(f"Boundary positions: {len(boundary_commits)}")
print(f"Elementary positions: {len(elementary_commits)}")
print(f"Boundary commit layers: {boundary_commits}")
print(f"Elementary commit layers: {elementary_commits}")
print(f"Mean boundary: {np.mean(boundary_commits):.1f}")
print(f"Mean elementary: {np.mean(elementary_commits):.1f}")

Boundary positions: 19
Elementary positions: 2
Boundary commit layers: [24, 23, 26, 28, 25, 25, 24, 24, 24, 25, 28, 24, 25, 24, 25, 27, 26, 27, 28]
Elementary commit layers: [25, 26]
Mean boundary: 25.4
Mean elementary: 25.5


In [26]:
import re

def find_numeric_positions(all_tokens, input_len):
    """Find all positions in the response where the model generates a number."""
    numeric_positions = []
    for i in range(input_len, len(all_tokens)):
        token = all_tokens[i].strip()
        # Check if token is purely numeric
        if token.isdigit() or re.match(r'^\d+$', token):
            numeric_positions.append({
                'position': i,
                'token': token
            })
    return numeric_positions

# Test on problem 0
numeric_positions = find_numeric_positions(all_tokens, input_len)
print(f"Total numeric positions in response: {len(numeric_positions)}")
print(f"\nFirst 20:")
for n in numeric_positions[:20]:
    pred = n['token']
    classification = 'ELEMENTARY' if pred in input_nums else 'BOUNDARY'
    print(f"  Position {n['position']}: '{pred}' → {classification}")

Total numeric positions in response: 54

First 20:
  Position 3883: '1' → ELEMENTARY
  Position 3893: '22' → ELEMENTARY
  Position 3896: '13' → ELEMENTARY
  Position 3899: '6' → ELEMENTARY
  Position 3903: '10' → ELEMENTARY
  Position 3926: '40' → BOUNDARY
  Position 3928: '2' → ELEMENTARY
  Position 3939: '44' → ELEMENTARY
  Position 3942: '22' → ELEMENTARY
  Position 3945: '20' → ELEMENTARY
  Position 3949: '69' → ELEMENTARY
  Position 3978: '45' → BOUNDARY
  Position 3980: '3' → ELEMENTARY
  Position 3991: '34' → ELEMENTARY
  Position 3994: '18' → ELEMENTARY
  Position 3997: '12' → ELEMENTARY
  Position 4001: '51' → ELEMENTARY
  Position 4029: '150' → BOUNDARY
  Position 4031: '4' → ELEMENTARY
  Position 4041: '38' → ELEMENTARY


In [27]:
boundary_commits = []
elementary_commits = []

for n in numeric_positions:
    pos = n['position']
    token = n['token']
    commit = find_commit_layer(pos, layer_predictions, layer_probs)
    if token in input_nums:
        elementary_commits.append(commit)
    else:
        boundary_commits.append(commit)

print(f"Boundary positions: {len(boundary_commits)}")
print(f"Elementary positions: {len(elementary_commits)}")
print(f"Mean boundary commit layer: {np.mean(boundary_commits):.1f}")
print(f"Mean elementary commit layer: {np.mean(elementary_commits):.1f}")
print(f"Boundary commits: {boundary_commits}")
print(f"Elementary commits: {elementary_commits}")

Boundary positions: 27
Elementary positions: 27
Mean boundary commit layer: 27.8
Mean elementary commit layer: 28.3
Boundary commits: [31, 27, 28, 31, 31, 28, 28, 28, 21, 30, 30, 30, 28, 31, 30, 30, 30, 32, 19, 22, 21, 23, 22, 31, 27, 30, 32]
Elementary commits: [31, 25, 28, 23, 28, 31, 28, 28, 24, 30, 31, 28, 30, 24, 30, 31, 28, 30, 25, 30, 31, 27, 28, 24, 29, 31, 32]


In [28]:
boundary_commits = []
elementary_commits = []

for n in numeric_positions:
    pos = n['position']
    token = n['token']

    # Measure at N-1 — that's where the decision to write this token was made
    decision_pos = pos - 1

    commit = find_commit_layer(decision_pos, layer_predictions, layer_probs)
    if token in input_nums:
        elementary_commits.append(commit)
    else:
        boundary_commits.append(commit)

print(f"Boundary positions: {len(boundary_commits)}")
print(f"Elementary positions: {len(elementary_commits)}")
print(f"Mean boundary commit layer: {np.mean(boundary_commits):.1f}")
print(f"Mean elementary commit layer: {np.mean(elementary_commits):.1f}")
print(f"Boundary commits: {boundary_commits}")
print(f"Elementary commits: {elementary_commits}")

Boundary positions: 27
Elementary positions: 27
Mean boundary commit layer: 25.1
Mean elementary commit layer: 24.5
Boundary commits: [24, 23, 26, 28, 24, 24, 24, 23, 25, 24, 24, 24, 28, 25, 24, 24, 24, 25, 28, 24, 25, 24, 25, 27, 26, 27, 28]
Elementary commits: [32, 27, 23, 23, 24, 29, 23, 23, 23, 24, 27, 24, 23, 23, 23, 24, 24, 23, 23, 24, 28, 24, 23, 23, 24, 25, 26]


In [31]:
import torch
from scipy import stats

# ── MANUAL RANGE CONTROL ──────────────────────────────────────────
START_IDX = 0
END_IDX = 60
# ──────────────────────────────────────────────────────────────────

LOGIT_LENS_PATH = '/content/logit_lens_llama_3.1_8b.json'

def get_logit_lens(full_text, model, tokenizer):
    inputs = tokenizer(full_text, return_tensors="pt").to(model.device)
    total_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    hidden_states = outputs.hidden_states
    final_layer_norm = model.model.norm
    lm_head = model.lm_head

    layer_predictions = []
    layer_probs = []

    for layer_idx in range(1, 33):
        hs = hidden_states[layer_idx]

        with torch.no_grad():
            normed = final_layer_norm(hs)
            logits = lm_head(normed)

        probs = torch.softmax(logits[0], dim=-1)
        top_tokens = torch.argmax(probs, dim=-1)
        top_probs = probs[range(len(top_tokens)), top_tokens]

        pred_strings = [tokenizer.decode([t.item()]) for t in top_tokens]
        prob_values = [p.item() for p in top_probs]

        layer_predictions.append(pred_strings)
        layer_probs.append(prob_values)

        del normed, logits, probs, top_tokens, top_probs
        torch.cuda.empty_cache()

    del outputs, hidden_states
    torch.cuda.empty_cache()

    all_tokens = [tokenizer.decode([t.item()]) for t in inputs["input_ids"][0]]

    return all_tokens, layer_predictions, layer_probs, total_len

def build_prompt_text(problem):
    system = (
        "You are an airline pricing assistant. You will be given the policies of "
        "American Airlines and a passenger's itinerary and baggage details. "
        "Compute the TOTAL COST (flight ticket + all checked bag fees, including "
        "any oversize or overweight charges) according to the policies. "
        "Show your step-by-step reasoning for each bag, then give the final answer "
        "as: FINAL TOTAL: $<amount>"
    )
    user = f"POLICIES:\n{REFERENCE_RULES}\n\nPASSENGER:\n{problem['prompt']}"
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    return tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )

# Load existing if resuming
if os.path.exists(LOGIT_LENS_PATH):
    try:
        with open(LOGIT_LENS_PATH) as f:
            logit_lens_results = json.load(f)
        done_indices = {r["problem_idx"] for r in logit_lens_results}
        print(f"Resuming: {len(logit_lens_results)} problems done.")
    except:
        os.remove(LOGIT_LENS_PATH)
        logit_lens_results = []
        done_indices = set()
        print("Corrupted file - starting fresh.")
else:
    logit_lens_results = []
    done_indices = set()
    print("Starting fresh.")

for idx in range(START_IDX, END_IDX):
    if idx in done_indices:
        continue

    print(f"\n--- Logit Lens Problem {idx+1}/{END_IDX} ---")
    torch.cuda.empty_cache()

    try:
        problem = problems[idx]
        result = results[idx]
        input_nums = get_input_numbers(problem)

        prompt_text = build_prompt_text(problem)
        prompt_tokens = tokenizer(prompt_text, return_tensors="pt")
        input_len = prompt_tokens["input_ids"].shape[-1]

        full_text = prompt_text + result['condition_a_response']

        all_tokens, layer_predictions, layer_probs, total_len = get_logit_lens(full_text, model, tokenizer)

        numeric_positions = find_numeric_positions(all_tokens, input_len)

        boundary_commits = []
        elementary_commits = []
        position_data = []

        for n in numeric_positions:
            pos = n['position']
            token = n['token']
            decision_pos = pos - 1
            commit = find_commit_layer(decision_pos, layer_predictions, layer_probs)
            classification = 'elementary' if token in input_nums else 'boundary'

            if classification == 'elementary':
                elementary_commits.append(commit)
            else:
                boundary_commits.append(commit)

            position_data.append({
                'position': pos,
                'token': token,
                'classification': classification,
                'commit_layer': commit
            })

        # Free memory after processing each problem
        del all_tokens, layer_predictions, layer_probs
        torch.cuda.empty_cache()

        logit_lens_results.append({
            'problem_idx': idx,
            'boundary_commits': boundary_commits,
            'elementary_commits': elementary_commits,
            'position_data': position_data,
            'n_boundary': len(boundary_commits),
            'n_elementary': len(elementary_commits),
            'mean_boundary': float(np.mean(boundary_commits)) if boundary_commits else None,
            'mean_elementary': float(np.mean(elementary_commits)) if elementary_commits else None,
        })

        with open(LOGIT_LENS_PATH, 'w') as f:
            json.dump(logit_lens_results, f, indent=2)

        print(f"  Boundary: {len(boundary_commits)} positions, mean commit layer {np.mean(boundary_commits):.1f}")
        print(f"  Elementary: {len(elementary_commits)} positions, mean commit layer {np.mean(elementary_commits):.1f}")
        print(f"  [Saved problem {idx+1}]")

    except Exception as e:
        print(f"  ERROR on problem {idx}: {e}")
        torch.cuda.empty_cache()
        continue

all_boundary = []
all_elementary = []
for r in logit_lens_results:
    all_boundary.extend(r['boundary_commits'])
    all_elementary.extend(r['elementary_commits'])

stat, pvalue = stats.mannwhitneyu(all_boundary, all_elementary, alternative='greater')

print(f"\n=== FINAL LOGIT LENS RESULTS ===")
print(f"Total boundary positions: {len(all_boundary)}")
print(f"Total elementary positions: {len(all_elementary)}")
print(f"Mean boundary commit layer: {np.mean(all_boundary):.2f}")
print(f"Mean elementary commit layer: {np.mean(all_elementary):.2f}")
print(f"Mann-Whitney U statistic: {stat:.1f}")
print(f"P-value: {pvalue:.4f}")
print(f"Saved to {LOGIT_LENS_PATH}")

Starting fresh.

--- Logit Lens Problem 1/60 ---
  ERROR on problem 0: CUDA out of memory. Tried to allocate 2.14 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.35 GiB is free. Including non-PyTorch memory, this process has 13.21 GiB memory in use. Of the allocated memory 10.86 GiB is allocated by PyTorch, and 2.22 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

--- Logit Lens Problem 2/60 ---
  ERROR on problem 1: CUDA out of memory. Tried to allocate 2.63 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.35 GiB is free. Including non-PyTorch memory, this process has 13.21 GiB memory in use. Of the allocated memory 10.93 GiB is allocated by PyTorch, and 2.15 GiB is reserved by PyTorch but unallocate

/tmp/ipykernel_1003/1406369470.py:165: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  stat, pvalue = stats.mannwhitneyu(all_boundary, all_elementary, alternative='greater')
